# Hyperspy Tutorial - Excite Workshop
### 03 / 12 / 2026 

####  Thomas Ginnis

The following notebook is a completed notebook demonstrating a basic EELS analysis workflow using Hyperspy, though it can be modified to process other hyperspectral datasets.

The workbook will demonstrate:
- How to load datasets into hyperspy
- How to plot and inspect HAADF images
- Basic EELS pre-processing (ZLP alignment, spikes removal)
- Thickness mapping with EELS
- Basic edge mapping with EELS
- Basic core-loss edge analysis

# Importing Libraries

First we import plotting libraries, and the hyperspy api.

Run the cell by clicking on it and pressing `Shift`+`Enter`

In [24]:
%matplotlib qt
#plotting library
import hyperspy.api as hs #hyperspy
import numpy as np # numpy - used for general data objects
import matplotlib.pyplot as plt # for more customised plotting
from matplotlib_scalebar.scalebar import ScaleBar # for scalebars in custom plots
from pathlib import Path

Verify hyperspy is installed correctly by determining the version

In [3]:
hs.__version__

'2.4.0'

# Loading Data

First define a path to our EELS dataset.

In [4]:
file_path = "hyperspy_data/STEM SI.dm4"

Our data is in the gatan `.dm4` file path, which is a common dataset for STEM EELS datasets. Hyperspy can load it using the `hs.load` function.

Do not worry about the warnings - the data has loaded fine

In [5]:
data = hs.load(file_path)

C:\Users\ginni\miniconda3\envs\hyperspy_env\Lib\site-packages\hyperspy\misc\_utils.py:1590: VisibleDeprecationWarning: Importing `LazyComplexSignal1D` from `hyperspy._signals.complex_signal1d` is deprecated and will be removed in the HyperSpy 3.0 release. Import it from `hyperspy.signals` instead.
  warnings.warn(
C:\Users\ginni\miniconda3\envs\hyperspy_env\Lib\site-packages\hyperspy\misc\_utils.py:1590: VisibleDeprecationWarning: Importing `LazySignal1D` from `hyperspy._signals.signal1d` is deprecated and will be removed in the HyperSpy 3.0 release. Import it from `hyperspy.signals` instead.
  warnings.warn(


We can see what the data object looks like by running `data`

In [6]:
data

[<Signal2D, title: HAADF Image (SI Survey), dimensions: (|512, 512)>,
 <Signal2D, title: HAADF Image, dimensions: (|64, 63)>,
 <EELSSpectrum, title: EELS Spectrum Image (low-loss), dimensions: (64, 63|2048)>,
 <EELSSpectrum, title: EELS Spectrum Image (high-loss), dimensions: (64, 63|2048)>]

It is a list containing 4 signals.
    1. A HAADF survey image, that was used to define the region of the EELS scan
    2. A HAADF image, aquired simultaneously with the EELS dataset
    3. A low-loss EELS dataset
    4. A high-loss EELS dataset

Each signal has a dimension too. For example, the high loss eels has dimensions: `(64, 63|2048)`, meaning the dataset is 64x63 pixels, with each pixel containing a spectrum with 2048 energy channels.

This workshop will focus on the EELS data, so we will use these parts of our data for the rest of our analysis. It is convenient to define two new varaible names, using python list indexing (python starts its lists from 0)

In [15]:
s_survey = data[0]
s_haadf = data[1]
s_ll = data[2]
s_hl = data[3]

# Inspecting and plotting the data

## Inspecting the metadata

The metadat for a dataset can be inspecting using the `.metadata` of the signal object. (note - this is according to the micrsocope. If the metadata recorded by the microscope is inccorrect, it will still be incorrect here!)

In [13]:
s_hl.metadata

├── Acquisition_instrument
│   └── TEM
│       ├── Detector
│       │   └── EELS
│       │       ├── aperture_size = 5.0
│       │       ├── collection_angle = 100.0
│       │       ├── dwell_time = 0.09975062344139651
│       │       ├── frame_number = 1
│       │       └── spectrometer = Enfinium ER
│       ├── Stage
│       │   ├── tilt_alpha = 0.6095125024601697
│       │   ├── tilt_beta = 0.0
│       │   ├── x = -0.18691300000000002
│       │   ├── y = 0.170743
│       │   └── z = 0.00271212
│       ├── acquisition_mode = STEM
│       ├── beam_current = 0.0
│       ├── beam_energy = 200.0
│       ├── camera_length = 54.0
│       ├── convergence_angle = 0.0
│       ├── magnification = 20000.0
│       └── microscope = Osiris
├── General
│   ├── FileIO
│   │   └── 0
│   │       ├── hyperspy_version = 2.4.0
│   │       ├── io_plugin = rsciio.digitalmicrograph
│   │       ├── operation = load
│   │       └── timestamp = 2026-11-25T12:37:11.575846+00:00
│   ├── date = 2025-06-18
│   ├── original_filename = STEM SI.dm4
│   ├── time = 14:45:50
│   └── title = EELS Spectrum Image (high-loss)
└── Signal
    ├── Noise_properties
    │   └── Variance_linear_model
    │       ├── gain_factor = 0.2631579041481018
    │       └── gain_offset = 0.0
    ├── quantity = Electrons (Counts)
    └── signal_type = EELS

## Plotting the HAADF images

The signals can be quickly plotted using the `.plot()` function

In [17]:
s_survey.plot()
s_haadf.plot()

## Customised plotting and going further
We can extract the 'raw' data from this object using the .data command. It is a 2D array, with a number (haadf electrons count) at each pixel)

In [21]:
s_haadf.data

array([[2.4149095e+06, 2.4175460e+06, 2.4169310e+06, ..., 2.4230298e+06,
        2.4232678e+06, 2.4259148e+06],
       [2.4172855e+06, 2.4179695e+06, 2.4176048e+06, ..., 2.4233148e+06,
        2.4241828e+06, 2.4274018e+06],
       [2.4183442e+06, 2.4190568e+06, 2.4183945e+06, ..., 2.4227150e+06,
        2.4254315e+06, 2.4274652e+06],
       ...,
       [2.4557092e+06, 2.4447035e+06, 2.4404170e+06, ..., 2.4382432e+06,
        2.4397568e+06, 2.4401920e+06],
       [2.4495950e+06, 2.4413752e+06, 2.4376942e+06, ..., 2.4322290e+06,
        2.4332888e+06, 2.4357295e+06],
       [2.4460592e+06, 2.4390002e+06, 2.4378762e+06, ..., 2.4309558e+06,
        2.4315632e+06, 2.4328640e+06]], shape=(63, 64), dtype=float32)

To understand more about the axis (eg, dimensions, pixel size, units) we use the `axes_manager` command.

In [22]:
s_haadf.axes_manager

Signal axis name,size,,offset,scale,units
x,64,,-0.0,0.011888236738741398,µm
y,63,,-0.0,0.011888236738741398,µm


We can index this axes manager object to get specific informataion, eg. the units of the x axis

In [23]:
s_haadf.axes_manager.signal_axes[0].units

'µm'

We can use these ways of interrogating the `s_haadf` data to build a cell that plots it automatically with a customised scale bar and colour scheme. 

(Don't worry too much about the specifics here if you are new to python and plotting with python - the code is just illustrative of how we can use thy hyperspy objects!)

In [81]:
data = s_haadf.data
ax_x = s_haadf.axes_manager.signal_axes[0]   # x axis (signal_axes[0] is x in HyperSpy)

# Percentile-based display limits
vmin, vmax = np.percentile(data, [1, 99])

fig, ax = plt.subplots(figsize=(6, 6)) # setting up a custom plot
ax.imshow(data, cmap="inferno", vmin=vmin, vmax=vmax) # using the inferno color map and the chosen percentiles for our maximum and minimum intensities

# Remove axes, ticks and labels
ax.axis("off")

# Automatic scale bar from the calibrated pixel size
scalebar = ScaleBar(
    ax_x.scale,
    units=ax_x.units,          
    location="lower right",
    box_alpha=0,               # transparent background box
    color="white",
    length_fraction=0.2,
)
ax.add_artist(scalebar)

plt.tight_layout() # narrow binding box


# Save to outputs/haadf_image.png at 600 dpi (create folder if needed)
out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "haadf_image.png", dpi=600, bbox_inches="tight", pad_inches=0)


# EELS data

## Initial Data inspection

First lets plot our EELS data to see what the datasets look like
Note how 2 windows pop up! The first showing our navigator which allows us to select a pixel, and the second showing an EELS signal for that pixel.

In [30]:
s_ll.plot()

### What does the navigator image correspond to?
By defauly - its simply the integral of the signal axis at each pixel. In this case, it doesnt provide us any useful contrast or meaningful chemical / thickness information. 

But we can manually set the navigator image in our plot to the HAADF image which was aquired simultaneosly!

In [35]:
s_ll.navigator = s_haadf

In [36]:
s_ll.plot()

We can do the same for the high loss dataset

In [37]:
s_hl.navigator = s_haadf

In [38]:
s_hl.plot()

## Aligning the ZLP

When navigating the low loss image - note how the ZLP is not always at the 0 eV energy channel (an artefact resulting from scanning and rastering the electron beam). 

The first step in processing an EELS dataset is aligning this sharp ZLP to the energy chanell by shifting the whole signal. 

We also should shift our high loss signal that was aquired simultaneosuly. 

To do this, we use the `align_zero_loss_peak` function. We can first check what arguments it expects using the `?` 

In [39]:
s_ll.align_zero_loss_peak?

Signature:
s_ll.align_zero_loss_peak(
    calibrate=True,
    also_align=[],
    print_stats=True,
    subpixel=True,
    mask=None,
    signal_range=None,
    show_progressbar=None,
    crop=True,
    **kwargs,
)
Docstring:
Align the zero-loss peak.

This function first aligns the spectra using the result of
`estimate_zero_loss_peak_centre` which finds the maximum in the
given energy range, then if subpixel is True,
proceeds to align with subpixel accuracy using `align1D`. The offset
is automatically correct if `calibrate` is True.

Parameters
----------
calibrate : bool
    If True, set the offset of the spectral axis so that the
    zero-loss peak is at position zero.
also_align : list of signals
    A list containing other spectra of identical dimensions to
    align using the shifts applied to the current spectrum.
    If `calibrate` is True, the calibration is also applied to
    the spectra in the list.
print_stats : bool
    If True, print summary statistics of the ZLP maximum 

In [40]:
s_ll.align_zero_loss_peak(signal_range=[-5.0,5.0],also_align=[s_hl])

Initial ZLP position statistics
-------------------------------
Summary statistics
------------------
mean:	3.32
std:	0.551

min:	2
Q1:	3
median:	3.5
Q3:	4
max:	4.5


  0%|          | 0/271 [00:00<?, ?it/s]

  0%|          | 0/271 [00:00<?, ?it/s]

  0%|          | 0/181 [00:00<?, ?it/s]

  0%|          | 0/271 [00:00<?, ?it/s]

  0%|          | 0/271 [00:00<?, ?it/s]

Now when we replot our signals, the ZLP should be aligned.

In [41]:
s_ll.plot()

## Spikes removal
Some EELS datasets contain spikes at individual energy loss channels arising from cosmic rays hitting the detector! Hyperspy has a built in tool 

In [42]:
s_ll.spikes_removal_tool()

In [43]:
s_hl.spikes_removal_tool()

Fortunatley - our dataset does not contain any such spikes. If you notice significant spikes at single energy channels, then you will need to process them using this tool (but be careful not to remove any real features like the ZLP!)

## Relative thickness maps

It may be useful to estimate the thickness of our sample. This can be done using the following hyperspy method, which derives the relative thickness in terms of the electron mean free path (which is energy and sample dependent) based on Poisson statistics. It requires the ZLP to be isolated.

For more information on the thickness estimation (or any other!) function in hyperspy, and the specifics of how it works, use the documentation.
https://hyperspy.org/exspy/reference/signals.html#exspy.signals.EELSSpectrum.estimate_thickness


If we want to quantify the absolute thickness, we will need to know the material density and this MFP, but for now we will just calculate relative thickness. But these arguments can be input into the `estimate_thickness` function if they are known for your material,

In [48]:
s_ll_zlp.plot()

In [49]:
thickness = s_ll.estimate_thickness(zlp=s_ll_zlp)

Computing the thickness without taking into account the effect of the limited collection angle, what usually leads to underestimating the thickness. To perform the angular corrections you must provide the density of the material.
Computing the relative thickness. To compute the absolute thickness provide the `mean_free_path` and/or the `density`


In [51]:
thickness.plot()

Our sample is reasonably uniform thickness, with the sulfide particle having a higher relative thickness. In general, it is approximatley 0.8-1.2 MFP thick, which is suitable for core-loss EELS analysis.

## Elemental Mapping

The areal density of an element at a pixel is proportional to the integral of its core-loss edge using EELS. This allows us to produce maps showing the elemental distribution.

We can use this to map the S_L23 edge in this sample, at 165 eV. First we can inspect this edge by isolating this region of the spectrum, using the `.isig` feature.

In [53]:
spectrum_range = 100.0 #energy range we want to plot
edge_energy =165.0

lower = edge_energy - 0.5*spectrum_range
upper = edge_energy + 0.5*spectrum_range

In [58]:
s_S = s_hl.isig[lower:upper]
s_S.navigator = s_haadf

In [56]:
s_S.plot()

## Background removal

The first step is removing the pre-edge background modelled using a power law. We can estimate the regions to fit this pre-edge region using the `background_removal` gui

In [59]:
s_S.remove_background()

It looks like 130-160 is a good window to choose.

In [60]:
s_S_bg_removed = s_S.remove_background(signal_range=[130.,160.],fast=False,background_type='PowerLaw')

100%|██████████| 4032/4032 [00:09<00:00, 406.76it/s]


In [61]:
s_S_bg_removed.plot()

## Saving data at intermediate points

We have now done a lot of processing to this data! We have
- Performed ZLP alignment
- Removed cosmic ray spikes
- Isolated the S edge
- Performed background removal

It might be a good idea to save this background removed dataset (if our PC crashes we will need to do it all again!)
We should save our aligned low loss data too.

We do that with the save function:

In [65]:
s_S_bg_removed.save('outputs/s_S_bg_removed.hspy')
s_ll.save('outputs/s_ll_aligned.hspy')

If our machine crashes, we can load it back using `hs.load` again. (after we have reloaded the libraries at the top of the page!)


In [64]:
s_S_bg_removed = hs.load('outputs/S_L_bg_removed.hspy')

# Making a map

Since we removed our background, our new navigator is already close to a S map! But we can define a specific integration window and use this to create our map.
(If we are mapping and comparing multiple edges in a semi-quantitative manner, we will need to be careful and consistent about the window we use)

Again, we can use `.isig` to define the energy window, then integrate1D to perform the integration and make a map.

In [66]:
integration_window = 40.0 # 40eV is a typical window width to use

s_S_isolated = s_S_bg_removed.isig[edge_energy:edge_energy+integration_window]

In [71]:
S_map = s_S_isolated.integrate1D(axis='Energy loss')

In [73]:
S_map.plot()

As before, we can use a customised script to make a customised plot.



In [75]:
data = S_map.data
ax_x = S_map.axes_manager.navigation_axes[0]   # x axis (signal_axes[0] is x in HyperSpy)

# Percentile-based display limits
vmin, vmax = np.percentile(data, [1, 99])

fig, ax = plt.subplots(figsize=(6, 6)) # setting up a custom plot
ax.imshow(data, cmap="inferno", vmin=vmin, vmax=vmax) # using the inferno color map and the chosen percentiles for our maximum and minimum intensities

# Remove axes, ticks and labels
ax.axis("off")

# Automatic scale bar from the calibrated pixel size
scalebar = ScaleBar(
    ax_x.scale,
    units=ax_x.units,          
    location="lower right",
    box_alpha=0,               # transparent background box
    color="white",
    length_fraction=0.2,
)
ax.add_artist(scalebar)

plt.tight_layout() # narrow binding box


out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "S_map.png", dpi=600, bbox_inches="tight", pad_inches=0)


## Core edge analysis

Now we may want to inspect the S edge in more detail. For this, it is useful to sum pixels in the same grain / region of the scan, to reduce the noise on the summed spectra.

We can use the region of interest (ROI) tools in hyperspy to perform this function.

First- it will be useful to use our S map as our navigator, and use the background removed S signal.



In [82]:
s_S_bg_removed.navigator = S_map

In [83]:
s_S_bg_removed.plot()

Defining a custom shaped ROI from the hyperspy library.

In [84]:
roi1 = hs.roi.PolygonROI()

Let us add this roi object to the low loss and high loss plots (the reason for including it in the low loss plot will become clear when we perform deconvolution)

In [88]:
s_S_bg_removed.plot()
s_ll.plot()
roi1.add_widget(s_S_bg_removed)
roi1.add_widget(s_ll)

Now in the navigator gui, we can draw a custom roi around the S rich grain.

**Leave the plot windows open for the next steps!**

Now we can extract the spectra from these regions of interest with the follwing command

In [89]:
S_roi1_hl = roi1.interactive(s_S_bg_removed)
S_roi1_ll = roi1.interactive(s_ll)

Lets plot these to see what they look like

In [87]:
S_roi1_hl.plot()

Now lets get the summed spectra

Due to the polygon shapes - we need to remove any Non -spectra pixels (from outside the ROI) in our sum.




In [92]:
S_hl_sum1 = S_roi1_hl.nansum()
S_ll_sum1 = S_roi1_ll.nansum()

In [93]:
S_hl_sum1.plot()

## Deconvolution

The spectra we record is the true single scattering spectrum, convolved with the low loss spectrum. This increases the intensity of features approximatley 20 eV after the edge onset.

Hyperspy has various tools for dealing with this. A simple, but effective, method is fourier ratio deconvolution (details can be found in the documentation).

In [94]:
S_hl_sum1.fourier_ratio_deconvolution?

Signature:
S_hl_sum1.fourier_ratio_deconvolution(
    low_loss,
    fwhm=None,
    threshold=None,
    extrapolate_lowloss=True,
    extrapolate_coreloss=True,
)
Docstring:
Performs Fourier-ratio deconvolution.

The core-loss should have the background removed. To reduce the noise
amplification the result is convolved with a Gaussian function.

Parameters
----------
low_loss : EELSSpectrum
    The corresponding low-loss EELSSpectrum.
fwhm : float or None
    Full-width half-maximum of the Gaussian function by which
    the result of the deconvolution is convolved. It can be
    used to select the final SNR and spectral resolution. If
    None, the FWHM of the zero-loss peak of the low-loss is
    estimated and used.
threshold : {None, float}
    Truncation energy to estimate the intensity of the
    elastic scattering. If None the threshold is taken as the
    first minimum after the ZLP centre.
extrapolate_lowloss, extrapolate_coreloss : bool
    If True the signals are extrapolated u

Here we see we need the low loss spectrum, which is why we included it in our ROIs. Lets perform the deconvolution.


In [96]:
S_1_deconvolved = S_hl_sum1.fourier_ratio_deconvolution(low_loss=S_ll_sum1)

  0%|          | 0/2 [00:00<?, ?it/s]

In [97]:
S_1_deconvolved.plot()

Now is another good time to save the data:

In [98]:
S_1_deconvolved.save('outputs/S_1_deconvolved.hspy')

## Repeating these steps for the other sulfide grains
Using these same steps, we can repeat this for the neighbouring sulfide grain.

First define a new ROI

In [99]:
roi2 = hs.roi.PolygonROI()

Then render the plots and add the ROI, and extract and sum the pixels

In [101]:
s_S_bg_removed.plot()
s_ll.plot()
roi2.add_widget(s_S_bg_removed)
roi2.add_widget(s_ll)

In [102]:
S_roi2_hl = roi2.interactive(s_S_bg_removed)
S_roi2_ll = roi2.interactive(s_ll)

In [105]:
S_hl_sum2 = S_roi2_hl.nansum()
S_ll_sum2 = S_roi2_ll.nansum()

Now do the deconvolution

In [106]:
S_2_deconvolved = S_hl_sum2.fourier_ratio_deconvolution(low_loss=S_ll_sum2)

  0%|          | 0/2 [00:00<?, ?it/s]

Save the new spectrum

In [107]:
S_2_deconvolved.save('outputs/S_2_deconvolved.hspy')

## Plotting multiple ROIS

We can use the `plot_spectra` command to plot the 2 proccessed spectra from the 2 rois to compare them.


In [108]:
spectra_list=[S_1_deconvolved,S_2_deconvolved]

In [109]:
hs.plot.plot_spectra?

Signature:
hs.plot.plot_spectra(
    spectra,
    style='overlap',
    color=None,
    linestyle=None,
    drawstyle='default',
    padding=1.0,
    legend=None,
    legend_picking=True,
    legend_loc='upper right',
    fig=None,
    ax=None,
    auto_update=None,
    normalise=False,
    **kwargs,
)
Docstring:
Plot several spectra in the same figure.

Parameters
----------
spectra : list of :class:`~.api.signals.Signal1D` or :class:`~.api.signals.BaseSignal`
    Ordered spectra list of signal to plot. If `style` is "cascade" or
    "mosaic", the spectra can have different size and axes.
    For  :class:`~.api.signals.BaseSignal` with navigation dimensions 1
    and signal dimension 0, the signal will be transposed to form a
    :class:`~.api.signals.Signal1D`.
style : {``'overlap'`` | ``'cascade'`` | ``'mosaic'`` | ``'heatmap'``}, default 'overlap'
    The style of the plot: 'overlap' (default), 'cascade', 'mosaic', or
    'heatmap'.
color : None or (list of) matplotlib color, defaul

In [120]:
hs.plot.plot_spectra(spectra_list,style='cascade',normalise=True,legend=['Region 1', 'Region 2'],legend_loc='upper left')

<Axes: xlabel='Energy loss (eV)'>

If we want more control over axis limits (eg. for automation) or adding line markers

In [128]:
ax = hs.plot.plot_spectra(spectra_list, style='cascade', normalise=True,
                          legend=['Region 1', 'Region 2'], legend_loc='upper left')
fig = ax.figure
fig.set_size_inches(6, 5)

ax.set_xlim(152., 205.)

# Vertical marker line at 165 with a label
ax.axvline(165, color="k", linestyle="--", linewidth=1, alpha=0.7)
ax.text(165, 0.03, "S L$_{2,3}$", transform=ax.get_xaxis_transform(),
        ha="center", va="bottom", rotation=90, fontsize=10,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))

fig.tight_layout()
fig.savefig("outputs/spectra.png", dpi=600, bbox_inches="tight")
plt.show()